# 08 · Matplotlib과 Seaborn 시각화

그래프를 예쁘게 만드는 것보다 질문에 맞는 그래프를 고르고 축·단위·표본 수를 정확히 보여 주는 데 집중합니다.

> 위에서 아래로 실행하세요. 예제 데이터는 노트북 안에서 만듭니다. 코드 셀 아래의 출력으로 결과를 확인하고, 실제 데이터에서는 열 이름·단위·기간을 먼저 확인하세요.

## 그림과 축

**코드 → 코드 개념**: Figure는 전체 그림, Axes는 각 그래프의 좌표 영역이다.

**코드 사용법**: 설비별 온도 추세를 같은 축에 그린다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.DataFrame({"machine": ["A"] * 4 + ["B"] * 4,
                   "cycle": [1, 2, 3, 4] * 2,
                   "temperature": [70, 72, 75, 80, 69, 71, 74, 76],
                   "vibration": [2.0, 2.1, 2.5, 3.2, 1.8, 2.0, 2.2, 2.4]})
fig, ax = plt.subplots(figsize=(7, 3))
for name, part in df.groupby("machine"):
    ax.plot(part["cycle"], part["temperature"], marker="o", label=name)
ax.set(xlabel="Cycle", ylabel="Temperature (°C)", title="Temperature trend")
ax.legend(); ax.grid(alpha=0.3); fig.tight_layout(); plt.show()

**같은 결과를 얻는 방법과 선택 이유**

- `plt.plot(...)`는 빠른 한 장에 편하다. `fig, ax = plt.subplots()`는 여러 그래프·축을 명확하게 제어할 때 좋다.
- 시계열은 먼저 시간순 정렬한다. 선은 관측 사이가 이어진다는 느낌을 주므로 드문 관측이면 점도 함께 표시한다.

## 분포: 히스토그램과 상자그림

**코드 → 코드 개념**: 히스토그램은 빈도 분포, 상자그림은 그룹의 중앙값과 사분위 범위를 보여 준다.

**코드 사용법**: 같은 진동값을 두 관점으로 그린다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
sns.histplot(data=df, x="vibration", bins=5, ax=axes[0])
sns.boxplot(data=df, x="machine", y="vibration", ax=axes[1])
fig.tight_layout(); plt.show()

**같은 결과를 얻는 방법과 선택 이유**

- `plt.hist(df['vibration'])`는 배열만 있을 때 간단하다. `sns.histplot(data=df, x='vibration')`는 열 이름·범주 색을 붙이기 쉽다.
- `bins`에 따라 모양이 달라 보일 수 있다. 상자그림의 수염 밖 점은 검토 후보이지 자동 삭제 대상이 아니다.

## 건수와 평균의 차이

**코드 → 코드 개념**: 막대 높이가 건수인지 평균인지 구분해야 한다.

**코드 사용법**: 설비별 행 수와 평균 진동을 나란히 그린다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
sns.countplot(data=df, x="machine", ax=axes[0])
sns.barplot(data=df, x="machine", y="vibration", errorbar=None, ax=axes[1])
axes[0].set_title("Count"); axes[1].set_title("Mean vibration")
fig.tight_layout(); plt.show()

**같은 결과를 얻는 방법과 선택 이유**

- `countplot`은 행 수를 센다. `barplot`은 기본적으로 평균을 계산한다. 이름이 비슷해도 값이 다르다.
- 이미 집계된 표가 있다면 `ax.bar(summary.index, summary['mean'])`로 막대 높이를 명시하는 편이 안전하다.

## 산점도와 상관행렬

**코드 → 코드 개념**: 산점도는 개별 관측의 모양, 상관행렬은 수치 관계의 요약을 보여 준다.

**코드 사용법**: 두 센서의 관계를 점과 색으로 확인한다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
sns.scatterplot(data=df, x="temperature", y="vibration", hue="machine", ax=axes[0])
corr = df[["temperature", "vibration"]].corr()
sns.heatmap(corr, annot=True, vmin=-1, vmax=1, cmap="coolwarm", ax=axes[1])
fig.tight_layout(); plt.show()

**같은 결과를 얻는 방법과 선택 이유**

- `plt.scatter(x, y)`는 단순 두 배열에 편하다. `sns.scatterplot`은 `hue`로 설비별 패턴을 보기 좋다.
- 상관행렬의 큰 값은 관계 후보일 뿐이다. 시간 추세나 설비 차이를 분리해 보고 인과관계로 단정하지 않는다.

## 저장과 표시

**코드 → 코드 개념**: `savefig`는 그림을 저장하고 `show`는 화면에 표시한다.

**코드 사용법**: 메모리 버퍼에 PNG를 저장해 생성 여부를 확인한다.

In [ ]:
from io import BytesIO
fig, ax = plt.subplots(figsize=(4, 2))
ax.plot([1, 2, 3], [2, 3, 4])
fig.tight_layout()
buffer = BytesIO()
fig.savefig(buffer, format="png", dpi=150, bbox_inches="tight")
print("PNG bytes:", len(buffer.getvalue()))
plt.close(fig)

**같은 결과를 얻는 방법과 선택 이유**

- 실제 파일에는 `fig.savefig('trend.png', dpi=150, bbox_inches='tight')`를 `show()` 전에 호출한다. 메모리 버퍼는 디스크 파일 없이 저장 코드를 검증할 때 좋다.
- 한글 글꼴은 실행 환경마다 다르므로 원본의 글꼴 설정 노트북을 참고한다.

## 원본 학습 자료

[`4. Summary/08_matplotlib`](../4.%20Summary/08_matplotlib), [`1. lecture/02_Pandas/matplotlib`](../1.%20lecture/02_Pandas/matplotlib)